# AUROC Threshold & Hidden Layer Analysis

This notebook analyzes AUROC error detection across varying correctness thresholds and hidden layer ensemble configurations for all models.

### Baselines Evaluated:
1. **AU (entropy_mixture)** - Ensembled multi-layer predictive entropy (Aleatoric Uncertainty)
2. **Epistemic EU (mutual_info)** - Layerwise mutual information (Epistemic Uncertainty)
3. **AU (entropy_final)** - Single-layer token entropy baseline
4. **SDLG Baseline SE** - Standard SDLG semantic entropy with token-level debiasing mask
5. **Our Weighted SE** - Sentence-importance weighted semantic entropy (`seq_level_impo`)
6. **Kuhn Semantic Entropy (SE)** - Unweighted Monte Carlo cluster average baseline
7. **TokUR Epistemic (EU) & Aleatoric (AU)** - Token uncertainty routing baselines
8. **ICE Epistemic (EU), Aleatoric (AU), Total (TU)** - Internal consistency ensembling baselines


In [ ]:
import os
import glob
import pickle as pkl
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# -------------------------------------------------------------
# 1. Dynamic Path Resolution
# -------------------------------------------------------------
script_dir = os.path.dirname(os.path.abspath('__file__')) if '__file__' in locals() else os.getcwd()
project_root = os.path.abspath(os.path.join(script_dir, "..", "..")) if "src" in script_dir else os.path.abspath(os.path.join(script_dir, ".."))

candidate_dirs = [
    os.path.join(script_dir, "result", "entropies2"),
    os.path.join(script_dir, "entropies2"),
    os.path.join(project_root, "experiments", "result", "entropies2"),
    os.path.join(project_root, "result", "entropies2"),
    os.path.join(os.getcwd(), "result", "entropies2"),
    os.path.join(os.getcwd(), "experiments", "result", "entropies2"),
]

entropy_dir = None
for d in candidate_dirs:
    if os.path.exists(d) and glob.glob(os.path.join(d, "*_entropies_*.pkl")):
        entropy_dir = d
        break

if entropy_dir is None:
    entropy_dir = candidate_dirs[0]

candidate_tokur_dirs = [
    os.path.join(script_dir, "tokur", "result"),
    os.path.join(project_root, "experiments", "tokur", "result"),
    os.path.join(project_root, "results", "tokur"),
    os.path.join(project_root, "tokur", "result"),
    os.path.join(os.getcwd(), "experiments", "tokur", "result"),
    os.path.join(os.getcwd(), "tokur", "result"),
]

candidate_ice_dirs = [
    os.path.join(script_dir, "ice", "result"),
    os.path.join(project_root, "experiments", "ice", "result"),
    os.path.join(project_root, "results", "ice"),
    os.path.join(project_root, "ice", "result"),
    os.path.join(os.getcwd(), "experiments", "ice", "result"),
    os.path.join(os.getcwd(), "ice", "result"),
]

print(f"Active Entropy Directory: {entropy_dir}")

In [ ]:
# -------------------------------------------------------------
# 2. AUROC vs Correctness Threshold Sweeps
# -------------------------------------------------------------
def _model_aliases(model):
    m = model.lower()
    aliases = [m]
    if 'qwen-1.5b' in m: aliases.append(m.replace('qwen-1.5b', 'qwen2.5-1.5b'))
    elif 'qwen2.5-1.5b' in m: aliases.append(m.replace('qwen2.5-1.5b', 'qwen-1.5b'))
    if 'qwen2.5-3b' in m: aliases.append(m.replace('qwen2.5-3b', 'qwen-3b'))
    elif 'qwen-3b' in m: aliases.append(m.replace('qwen-3b', 'qwen2.5-3b'))
    if 'llama-3.2-3b' in m: aliases.append('llama3.2-3b')
    elif 'llama3.2-3b' in m: aliases.append('llama-3.2-3b')
    return list(set(aliases))

def _dataset_aliases(dataset):
    d = dataset.lower()
    aliases = [d]
    if d == 'truth' or d == 'truthful': aliases.extend(['truth', 'truthful', 'truthfulqa'])
    elif d == 'science' or d == 'scienceqa': aliases.extend(['science', 'scienceqa', 'sciq'])
    elif d == 'open' or d == 'openbook': aliases.extend(['open', 'openbook', 'openbookqa'])
    return list(set(aliases))

def load_run_metrics(entropy_file):
    label_file = entropy_file.replace("_entropies_", "_label_")
    question_file = entropy_file.replace("_entropies_", "_question_")
    if not os.path.exists(label_file): return None

    try:
        with open(entropy_file, 'rb') as f: raw_entropies = pkl.load(f)
        with open(label_file, 'rb') as f: labels = pkl.load(f)
        questions = pkl.load(open(question_file, 'rb')) if os.path.exists(question_file) else []
    except Exception as e:
        print(f"Error loading {entropy_file}: {e}")
        return None

    num_instances = len(labels)
    if num_instances == 0 or len(raw_entropies) == 0: return None

    ours_entropies = raw_entropies[:num_instances]
    sample_keys = list(ours_entropies[0].keys())
    au_key = 'entropy_final'
    if au_key not in ours_entropies[0]:
        au_candidates = [k for k in sample_keys if 'entropy_final' in k]
        if au_candidates: au_key = au_candidates[0]

    df = pd.DataFrame({
        'Question': questions[:num_instances] if len(questions) >= num_instances else range(num_instances),
        'Correctness': labels[:num_instances],
    'Our_Weighted_SE': [d.get('normalised_semantic_entropy', 0.0) for d in ours_entropies],
        'entropy_final': [d.get(au_key, 0.0) for d in ours_entropies],
        'entropy_mixture': [d.get('entropy_mixture', 0.0) for d in ours_entropies],
        'mutual_info': [d.get('mutual_info', 0.0) for d in ours_entropies],
    })

    # Segment 2 (N : 2N): SDLG Baseline SE
    if len(raw_entropies) >= 2 * num_instances:
        df['SDLG_Baseline_SE'] = [d.get('normalised_semantic_entropy', 0.0) for d in raw_entropies[num_instances : 2*num_instances]]
    else:
        df['SDLG_Baseline_SE'] = [d.get('normalised_semantic_entropy', 0.0) for d in ours_entropies]

    # Segment 3 (2N : 3N): Kuhn SE
    if len(raw_entropies) >= 3 * num_instances:
        df['Kuhn_SE'] = [d.get('normalised_semantic_entropy', 0.0) for d in raw_entropies[2*num_instances : 3*num_instances]]
    else:
        df['Kuhn_SE'] = np.nan

    fname = os.path.basename(entropy_file).replace(".pkl", "")
    parts = fname.split("_entropies_")
    if len(parts) == 2:
        meta = parts[1].replace("_new", "").split("_")
        dataset_name, model_name = meta[0], "_".join(meta[1:])
    else:
        dataset_name, model_name = "unknown", fname

    # TokUR
    m_aliases = _model_aliases(model_name)
    d_aliases = _dataset_aliases(dataset_name)

    tokur_found = False
    for tokur_dir in candidate_tokur_dirs:
        if tokur_found or not os.path.exists(tokur_dir): continue
        for folder in os.listdir(tokur_dir):
            fl = folder.lower()
            if any(ma in fl for ma in m_aliases) and any(da in fl for da in d_aliases):
                ef = os.path.join(tokur_dir, folder, "entropies_tokur.pkl")
                if os.path.exists(ef):
                    try:
                        with open(ef, "rb") as tf: tok_ent = pkl.load(tf)
                        n_t = len(tok_ent)
                        tok_au = [e.get('tokur_au', e.get('entropy_final', 0.0)) for e in tok_ent[:num_instances]]
                        tok_mi = [e.get('tokur_mi', e.get('mutual_info', 0.0)) for e in tok_ent[:num_instances]]
                        df['tokur_au'] = pd.array(tok_au + [None] * max(0, num_instances - n_t), dtype=object)
                        df['tokur_mi'] = pd.array(tok_mi + [None] * max(0, num_instances - n_t), dtype=object)
                        tokur_found = True
                        break
                    except Exception: pass

    # ICE
    ice_found = False
    for ice_dir in candidate_ice_dirs:
        if ice_found or not os.path.exists(ice_dir): continue
        for ice_f in os.listdir(ice_dir):
            fl = ice_f.lower()
            if (fl.endswith("_summary.pkl") or fl.endswith("_summary.csv")) and any(ma in fl for ma in m_aliases) and any(da in fl for da in d_aliases):
                fpath = os.path.join(ice_dir, ice_f)
                try:
                    ice_df = pd.read_csv(fpath) if fpath.endswith(".csv") else pd.DataFrame(pkl.load(open(fpath, "rb")))
                    n_i = len(ice_df)
                    def _pad(col): return pd.array(list(col.values[:num_instances]) + [None] * max(0, num_instances - n_i), dtype=object)
                    df['ice_epistemic'] = _pad(ice_df['epistemic_uncertainty'])
                    df['ice_aleatoric'] = _pad(ice_df['aleatoric_uncertainty'])
                    df['ice_total']     = _pad(ice_df['total_uncertainty'])
                    ice_found = True
                    break
                except Exception: pass

    return {'df': df, 'model': model_name, 'dataset': dataset_name, 'run_name': f"{model_name.upper()} ({dataset_name})", 'num_samples': num_instances, 'ours_entropies': ours_entropies}

entropy_files = sorted(glob.glob(os.path.join(entropy_dir, "*_entropies_*.pkl")))
candidate_thresholds = sorted([-0.70, -0.60, -0.50, -0.40, -0.35, -0.30, -0.20, -0.15, -0.10, -0.05, 0.0, 0.03, 0.07, 0.10, 0.13, 0.16, 0.20, 0.25, 0.30, 0.40])

# =========================================================================
# VISUAL DESIGN SYSTEM: High-contrast grouping of Ours vs. Baselines
# =========================================================================
metrics_config = {
    # --- Group 1: Ours (Thick Solid Lines, Vibrant Emerald & Indigo) ---
    'AU (entropy_mixture)':       {'col': 'entropy_mixture',  'color': '#059669', 'ls': '-',  'marker': 'o', 'lw': 2.8, 'zorder': 10}, # Emerald Green
    'Epistemic EU (mutual_info)': {'col': 'mutual_info',      'color': '#2563eb', 'ls': '-',  'marker': '^', 'lw': 2.8, 'zorder': 10}, # Royal Blue
    'Our Weighted SE':            {'col': 'Our_Weighted_SE',  'color': '#e11d48', 'ls': '-',  'marker': 'D', 'lw': 2.4, 'zorder': 9},  # Rose / Crimson

    # --- Group 2: Semantic Entropy Baselines (Dashed Lines, Amber & Violet) ---
    'SDLG Baseline SE':           {'col': 'SDLG_Baseline_SE', 'color': '#d97706', 'ls': '--', 'marker': 's', 'lw': 2.0, 'zorder': 8},  # Amber
    'Kuhn Semantic Entropy (SE)': {'col': 'Kuhn_SE',          'color': '#7c3aed', 'ls': '--', 'marker': 'x', 'lw': 2.0, 'zorder': 8},  # Violet / Purple

    # --- Group 3: Competing Decomposed Baselines (Dash-Dot, Teal, Cyan, Rust) ---
    'TokUR Epistemic (EU)':       {'col': 'tokur_mi',         'color': '#0d9488', 'ls': '-.', 'marker': 'P', 'lw': 1.8, 'zorder': 7},  # Teal
    'TokUR Aleatoric (AU)':       {'col': 'tokur_au',         'color': '#06b6d4', 'ls': '-.', 'marker': '*', 'lw': 1.8, 'zorder': 7},  # Cyan
    'ICE Epistemic (EU)':         {'col': 'ice_epistemic',    'color': '#c2410c', 'ls': '-.', 'marker': 'h', 'lw': 1.8, 'zorder': 6},  # Rust Orange
    'ICE Total (TU)':             {'col': 'ice_total',        'color': '#be185d', 'ls': '-.', 'marker': '+', 'lw': 1.8, 'zorder': 6},  # Magenta
    'ICE Aleatoric (AU)':         {'col': 'ice_aleatoric',    'color': '#b45309', 'ls': '-.', 'marker': 'd', 'lw': 1.8, 'zorder': 6},  # Sienna

    # --- Group 4: Single-Layer Baseline (Dotted Charcoal) ---
    'AU (entropy_final)':         {'col': 'entropy_final',    'color': '#475569', 'ls': ':',  'marker': 'v', 'lw': 1.8, 'zorder': 5},  # Slate Charcoal
}

for fpath in entropy_files:
    run_data = load_run_metrics(fpath)
    if run_data is None: continue
    df, run_name = run_data['df'], run_data['run_name']
    scores_arr = df['Correctness'].values

    valid_thresholds = [th for th in candidate_thresholds if len(np.unique((scores_arr < th).astype(int))) >= 2]
    if not valid_thresholds: continue

    results = {label: [] for label in metrics_config.keys()}
    for th in valid_thresholds:
        y_true = (scores_arr < th).astype(int)
        for label, cfg in metrics_config.items():
            col = cfg['col']
            if col in df.columns and df[col].notna().any():
                valid_mask = pd.notna(df[col].values) & (df[col].values != None)
                if np.sum(valid_mask) >= 5 and len(np.unique(y_true[valid_mask])) >= 2:
                    try:
                        score = roc_auc_score(y_true[valid_mask], np.array(df[col].values[valid_mask], dtype=float))
                    except Exception:
                        score = np.nan
                else:
                    score = np.nan
            else:
                score = np.nan
            results[label].append(score)

    plt.figure(figsize=(10, 6), dpi=120)
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    plt.grid(True, linestyle='--', alpha=0.5)

    for label, cfg in metrics_config.items():
        pairs = [(th, val) for th, val in zip(valid_thresholds, results[label]) if not np.isnan(val)]
        if pairs:
            xs, ys = zip(*pairs)
            plt.plot(xs, ys, label=label, color=cfg['color'], linestyle=cfg['ls'], marker=cfg['marker'], linewidth=cfg['lw'], markersize=6, alpha=0.9, zorder=cfg.get('zorder', 5))

    plt.axhline(0.5, color='gray', linestyle=':', label='Random Guess (0.50)', linewidth=1.2)
    plt.title(f"AUROC Performance vs. Correctness Threshold: {run_name}", fontsize=13, fontweight='bold', pad=12)
    plt.xlabel("Correctness Threshold (BLEURT Score)", fontsize=11, labelpad=8)
    plt.ylabel("AUROC Score (Higher is Better)", fontsize=11, labelpad=8)
    plt.ylim(0.35, 1.02)
    plt.legend(loc='upper left', bbox_to_anchor=(1.02, 1), borderaxespad=0, fontsize=9.5, frameon=True, shadow=True)
    plt.tight_layout()
    plt.show()

In [ ]:
# -------------------------------------------------------------
# 3. Layerwise Hidden Layer AUROC Sweep Across Thresholds
# -------------------------------------------------------------
for fpath in entropy_files:
    run_data = load_run_metrics(fpath)
    if run_data is None: continue
    df, run_name, ours_entropies = run_data['df'], run_data['run_name'], run_data['ours_entropies']
    scores_arr = df['Correctness'].values

    layers = sorted([int(m.group(1)) for k in ours_entropies[0].keys() if (m := re.match(r"^entropy_mixture_top_(\d+)$", k))])
    if not layers:
        layers = sorted([int(k.split('_')[-1]) for k in ours_entropies[0].keys() if k.startswith('entropy_mixture_') and k.split('_')[-1].isdigit()])
    if not layers: continue

    valid_thresholds = [th for th in candidate_thresholds if len(np.unique((scores_arr < th).astype(int))) >= 2]
    if not valid_thresholds: continue

    plt.figure(figsize=(11, 6.5), dpi=120)
    plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
    plt.grid(True, linestyle='--', alpha=0.5)

    colormap = plt.cm.coolwarm
    colors = colormap(np.linspace(0, 1, len(valid_thresholds)))

    plt.plot([], [], color='black', linestyle='-', marker='o', markersize=5, label='Entropy Mixture (Ours)')
    plt.plot([], [], color='black', linestyle='--', label='SDLG Baseline SE')
    plt.plot([], [], color='none', label=' ')

    for idx, thresh in enumerate(valid_thresholds):
        correct_labels = (scores_arr < thresh).astype(int)
        aurocs = []
        for L in layers:
            key = f"entropy_mixture_top_{L}" if f"entropy_mixture_top_{L}" in ours_entropies[0] else f"entropy_mixture_{L}"
            scores = [d.get(key, 0.0) for d in ours_entropies]
            aurocs.append(roc_auc_score(correct_labels, scores))

        base_scores = df['SDLG_Baseline_SE'].values
        base_auroc = roc_auc_score(correct_labels, base_scores)

        plt.plot(layers, aurocs, label=f"Thresh: {thresh}", color=colors[idx], linestyle='-', marker='o', markersize=5, linewidth=1.8, alpha=0.85)
        plt.plot([layers[0], layers[-1]], [base_auroc, base_auroc], color=colors[idx], linestyle='--', linewidth=1.4, alpha=0.7)

    plt.title(f"Entropy Mixture AUROC vs. Number of Ensembled Layers: {run_name}", fontsize=13, fontweight='bold', pad=12)
    plt.xlabel("Number of Top Hidden Layers (L)", fontsize=11, labelpad=8)
    plt.ylabel("AUROC Score (Higher is Better)", fontsize=11, labelpad=8)
    plt.xticks(layers)
    plt.ylim(0.35, 1.05)
    plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, shadow=True, fontsize=8.5, ncol=2)
    plt.tight_layout()
    plt.show()